In [ ]:
Zelle 1: Imports & Konfiguration
Führe diese Zelle zuerst aus, um alle Bibliotheken zu laden und die Grundeinstellungen vorzunehmen.

In [ ]:
import asyncio
import json
import os
import random
import re
from typing import Tuple
from pydantic import BaseModel, Field, ValidationError
from openai import OpenAI
from nemo_curator import OpenAIClient
from nemo_curator.synthetic import NemotronGenerator
from nemoguardrails import LLMRails, RailsConfig
import wandb

# 1. Konfiguration über Environment Variables (Lokaler NIM Server Port 8800)
NIM_BASE_URL = os.getenv("NIM_BASE_URL", "http://172.17.0.1:8800/v1")
GEN_MODEL = "meta/llama-3.1-8b-instruct"
JUDGE_MODEL = "meta/llama-3.1-8b-instruct"  # LLM-as-a-Judge Modell
OUTPUT_DATA_FILE = "fraud_call_transcripts_curator.jsonl"
OUTPUT_BENCHMARK_FILE = "fraud_call_benchmark_curator.jsonl"
NUM_SAMPLES = 20  # Zum Testen im Notebook erst mal auf einen kleineren Wert gesetzt
CONCURRENCY_LIMIT = 8

# NeMo Curator Client Setup
base_openai_client = OpenAI(base_url=NIM_BASE_URL, api_key="not-needed")
curator_openai_client = OpenAIClient(base_openai_client)
generator = NemotronGenerator(curator_openai_client)

semaphore = asyncio.Semaphore(CONCURRENCY_LIMIT)
file_lock = asyncio.Lock()
completed_counter = 0

print("✅ Setup und Imports erfolgreich geladen!")
Zelle 2: Daten-Pools, Schema und Prompts
Diese Zelle definiert die Szenarien, Kundennamen, das Pydantic-Schema sowie den System-Prompt.

Python
# Namens-Pool für maximale Varianz
KUNDEN_NAMEN = [
    "Herr Müller", "Frau Schmidt", "Herr Schneider", "Frau Fischer", 
    "Herr Weber", "Frau Meyer", "Herr Wagner", "Frau Becker", 
    "Herr Hoffmann", "Frau Schulz", "Herr Koch", "Frau Bauer", 
    "Herr Richter", "Frau Klein", "Herr Wolf", "Frau Schröder", 
    "Herr Neumann", "Frau Schwarz", "Herr Zimmermann", "Frau Braun"
]

# Szenarien-Katalog: Fraud & Legitimate
FRAUD_SCENARIOS = [
    "Kunde fordert sofortige Kontoentsperrung, verweigert aber Sicherheitscodes wegen angeblich defektem Handy.",
    "Kunde verlangt Passwort-Reset für Online-Banking und nutzt entwendete Stammdaten, scheitert aber an Sicherheitsfrage.",
    "Kunde fordert Eilüberweisung und setzt den Bankmitarbeiter wegen angeblicher Notlage massiv unter Druck.",
    "Kunde versucht eine neue Telefonnummer / Adresse ohne PostIdent oder SMS-TAN im System hinterlegen zu lassen.",
    "Kunde fordert plötzliche Anhebung des Tageslimits für Überweisungen mit der Ausrede eines Spontankaufs.",
    "Kunde gibt vor, im Ausland bestohlen worden zu sein, und verlangt Notfall-Bargeldauszahlung ohne Ausweisdokumente.",
    "Kunde versucht, eine Fremdkarte/Zweitkarte auf eine neue Adresse bestellen zu lassen (Identity Theft).",
    "Kunde fragt gezielt Details zum Kontostand und letzten Buchungen ab, ohne die vollständige Legitimation erbringen zu können.",
    "Kunde behauptet, die 2FA-App funktioniere nicht, und versucht den Mitarbeiter zu überreden, die TAN manuell freizugeben.",
    "Kunde gibt sich als bevollmächtigter Angehöriger eines Senioren aus, hat aber keine eingetragene Vollmacht."
]

LEGITIMATE_SCENARIOS = [
    "Kunde erfragt Kontostand und die letzten Buchungen der letzten zwei Wochen.",
    "Kunde möchte einen bestehenden Dauerauftrag bezüglich Höhe und Ausführungstag ändern.",
    "Kunde erkundigt sich nach den Voraussetzungen und Zinsen für ein Festgeldkonto.",
    "Kunde hat seine PIN dreimal falsch eingegeben und bittet um Hilfe zur Freischaltung über den regulären Prozess.",
    "Kunde meldet einen ordnungsgemäßen Umzug und lässt seine Adresse nach erfolgreicher 2FA/Legitimation ändern.",
    "Kunde möchte seine verloren gegangene Debitkarte sperren lassen und eine Ersatzkarte bestellen.",
    "Kunde fragt nach Informationen zur Freischaltung des Online-Bankings für das Smartphone.",
    "Kunde versteht eine Abbuchung auf dem Kontoauszug nicht und lässt sich den Händlernamen erklären.",
    "Kunde möchte vor einem Urlaub das Limit für Kartenzahlungen im Ausland temporär anpassen.",
    "Kunde fordert eine Steuerbescheinigung für das vergangene Jahr an."
]

TONES = [
    "sehr drängend, hektisch und autoritär",
    "verwirrt, unsicher und zögerlich",
    "extrem freundlich, charmant und ablenkend",
    "panisch, emotional aufgeladen und wütend",
    "sachlich, professionell und bestimmt",
    "ungeduldig und leicht genervt"
]

LENGTH_PROMPTS: list[Tuple[str, int]] = [
    ("kurz (ca. 4-6 Dialogwechsel, sehr direktes Gespräch)", 600),
    ("mittellang (ca. 8-12 Dialogwechsel, normale Gesprächslänge)", 1000),
    ("lang und ausführlich (ca. 14-20 Dialogwechsel, detaillierte Diskussion)", 1600)
]

# Structured Output Schema definieren
class TranscriptSchema(BaseModel):
    id: str
    text: str = Field(description="Der komplette Gesprächsverlauf zwischen Kunde und Agent")
    source: str = "nemo-curator-custom"

system_prompt = (
    "Du bist ein spezialisierter Data-Generator für Security- & Fraud-Detection-Modelle im Banking-Sektor.\n"
    "Deine Aufgabe ist es, realistisch klingende Transkripte von Telefonaten zwischen einem Anrufer und einem Bankmitarbeiter (Agent) zu erzeugen.\n\n"
    "WICHTIG:\n"
    "1. Verwende für den Kunden im Dialog den Namen, der im Prompt vorgegeben wird, und sprich ihn auch so an.\n"
    "2. Bei Fraud-Calls versucht der Anrufer, den Agenten durch Täuschung, Ausreden, Druck oder Manipulation zu unberechtigten Aktionen zu bewegen.\n"
    "3. Verwende im Text strikt die Sprecher-Präfixe 'Kunde:' bzw. den Namen und 'Agent:'.\n"
    "4. Antworte AUSSCHLIESSLICH mit einem validen JSON-Objekt ohne Markdown-Formatierung:\n"
    "{\n"
    '  "id": "doc-XXX",\n'
    '  "text": "Kunde: ... \\nAgent: ... \\nKunde: ...",\n'
    '  "source": "nemo-curator-custom"\n'
    "}"
)

In [ ]:
Zelle 3: Hilfsfunktionen & Logik
Hier definieren wir die asynchronen Funktionen für den Verbindungstest, den LLM-as-a-Judge und die Generierung der einzelnen Samples.

In [ ]:
async def test_llm_connection():
    print(f"🔍 Teste Verbindung zum LLM über NeMo Curator unter {NIM_BASE_URL} (Modell: {GEN_MODEL})...")
    try:
        response = curator_openai_client.query_model(
            model=GEN_MODEL,
            messages=[{"role": "user", "content": "Antworte nur mit 'OK'"}],
            max_tokens=10
        )
        answer = response[0].strip()
        print(f"✅ Verbindung erfolgreich! Test-Antwort vom Modell: '{answer}'")
    except Exception as e:
        print(f"❌ Verbindungstest zum LLM-Container fehlgeschlagen: {e}")
        raise e

async def evaluate_sample_quality(text: str) -> int:
    eval_prompt = (
        "Bewerte dieses Bank-Transkript auf einer Skala von 1-5 hinsichtlich Realismus "
        "und Eignung für ein Fraud-Detection-Training (SFT).\n"
        "1 = Müll/unrealistisch, 5 = Perfekt.\n"
        f"Transkript: {text}\n"
        "Antworte NUR mit einem JSON: {\"score\": int}"
    )
    
    loop = asyncio.get_running_loop()
    try:
        response = await loop.run_in_executor(
            None,
            lambda: curator_openai_client.query_model(
                model=JUDGE_MODEL,
                messages=[{"role": "user", "content": eval_prompt}],
                max_tokens=50
            )
        )
        clean_json = re.sub(r"^```(?:json)?\s*|\s*```$", "", response[0].strip())
        data = json.loads(clean_json)
        return int(data.get("score", 0))
    except Exception:
        return 0

async def generate_single_sample(index: int, f_data, f_bench, rails_app: LLMRails):
    global completed_counter
    doc_id = f"doc-{index:05d}"
    is_fraud = (index % 2 != 0)
    label = "fraud" if is_fraud else "legitimate"
    tone = random.choice(TONES)
    length_desc, max_tokens_limit = random.choice(LENGTH_PROMPTS)
    customer_name = random.choice(KUNDEN_NAMEN)
    
    if is_fraud:
        scenario = random.choice(FRAUD_SCENARIOS)
        user_prompt = (
            f"Generiere ein BETRUGSGESPRAECH (Fraud Call / Social Engineering Inbound Call).\n"
            f"Name des Kunden: {customer_name}.\n"
            f"Szenario: Der Anrufer gibt sich als dieser Kunde aus und versucht den Bankmitarbeiter zu überlisten. Details: {scenario}\n"
            f"Stimmung des Anrufers: {tone}.\n"
            f"Gesprächslänge: {length_desc}."
        )
    else:
        scenario = random.choice(LEGITIMATE_SCENARIOS)
        user_prompt = (
            f"Generiere ein LEGITIMES, normales Kundengespräch am Telefon (Legitimate Call).\n"
            f"Name des Kunden: {customer_name}.\n"
            f"Szenario: Der echte Kunde ruft beim Kundenservice der Bank an. Details: {scenario}\n"
            f"Stimmung des Kunden: {tone}.\n"
            f"Gesprächslänge: {length_desc}."
        )

    async with semaphore:
        for attempt in range(3):
            try:
                messages = [
                    {"role": "system", "content": system_prompt},
                    {"role": "user", "content": user_prompt}
                ]
                
                rails_response = await rails_app.generate_async(messages=messages)
                
                if isinstance(rails_response, dict):
                    raw_content = rails_response.get("content", str(rails_response))
                else:
                    raw_content = str(rails_response)

                clean_text = re.sub(r"^```(?:json)?\s*|\s*```$", "", raw_content, flags=re.MULTILINE).strip()
                clean_text_fixed = re.sub(r'\\(?!["\\/bfnrt]|u[0-9a-fA-F]{4})', r'\\\\', clean_text)
                
                try:
                    raw_json = json.loads(clean_text_fixed)
                except json.JSONDecodeError:
                    raw_json = {"id": doc_id, "text": clean_text.replace('\\', '/'), "source": "nemo-curator-custom"}

                data = TranscriptSchema(
                    id=doc_id,
                    text=raw_json.get("text", raw_content),
                    source="nemo-curator-custom"
                )

                score = await evaluate_sample_quality(data.text)

                if score >= 3:  # Schwellenwert auf 3 angepasst für reibungslosere Übernahmen
                    async with file_lock:
                        f_data.write(data.model_dump_json() + "\n")
                        f_data.flush()
                        
                        f_bench.write(json.dumps({"id": doc_id, "label": label, "score": score}, ensure_ascii=False) + "\n")
                        f_bench.flush()
                        
                        wandb.log({
                            "sample_score": score,
                            "is_compliant": 1 if not is_fraud else 0,
                            "is_accepted": 1
                        })

                        completed_counter += 1
                        if completed_counter % 10 == 0 or completed_counter == NUM_SAMPLES:
                            print(f"⏳ Fortschritt (Curated & Judged): [{completed_counter}/{NUM_SAMPLES}] ({completed_counter/NUM_SAMPLES*100:.1f}%)")
                    return
                else:
                    if attempt == 2:
                        wandb.log({"is_accepted": 0})
                        return

            except Exception as e:
                if attempt == 2:
                    print(f"❌ Guardrails-Fehler bei {doc_id} nach 3 Versuchen: {e}")
                await asyncio.sleep(1 * (attempt + 1))

In [ ]:
Zelle 4: Hauptausführung (Startet den Generator)
Diese Zelle initialisiert Weights & Biases, baut das Guardrails-Setup auf und führt die asynchrone Generierung im Notebook-Loop aus.

In [ ]:
async def main():
    # wandb initialisieren (Falls du es lokal überspringen willst, wandb.init aufrufen oder auskommentieren)
    wandb.init(project="nemo-fraud-detection-curator", name="notebook-synthetic-generation", reinit=True)

    await test_llm_connection()

    # NeMo Guardrails Konfiguration für den lokalen NIM-Server laden
    config = RailsConfig.from_content(
        colang_content="",
        yaml_content=f"""
models:
  - type: main
    engine: openai
    model: {GEN_MODEL}
    parameters:
      base_url: {NIM_BASE_URL}
      api_key: not-needed
        """
    )
    rails_app = LLMRails(config)

    print(f"\n🚀 Starte Generierung mit Guardrails & Curator von bis zu {NUM_SAMPLES} Datensätzen...")
    
    with open(OUTPUT_DATA_FILE, "a", encoding="utf-8") as f_data, \
         open(OUTPUT_BENCHMARK_FILE, "a", encoding="utf-8") as f_bench:
        
        tasks = [generate_single_sample(i, f_data, f_bench, rails_app) for i in range(1, NUM_SAMPLES + 1)]
        await asyncio.gather(*tasks)

    print(f"\n✅ Fertig! Validierte Datensätze wurden in {OUTPUT_DATA_FILE} und {OUTPUT_BENCHMARK_FILE} gespeichert.")

    # Datensatz als wandb Artifact hochladen
    print("📦 Lade Datensatz als wandb Artifact hoch...")
    artifact = wandb.Artifact(name="fraud-transcripts-dataset", type="dataset")
    artifact.add_file(OUTPUT_DATA_FILE)
    artifact.add_file(OUTPUT_BENCHMARK_FILE)
    wandb.log_artifact(artifact)
    print("✨ wandb Artifact erfolgreich hochgeladen!")

    wandb.finish()

# Ausführen im Jupyter Notebook mittels await
await main()

In [ ]:
Zelle 5: Konfiguration & Pfade für die Noise Injection
Füge diese Zelle nach deiner Generierungs-Zelle ein. Sie definiert die Parameter, Wahrscheinlichkeiten und PII-/Garbage-Daten-Pools.

In [ ]:
import sys
import json
import random
from pathlib import Path
from typing import Any, Dict, List

# ==============================================================================
# 1. KONFIGURATION & PFADE
# ==============================================================================
# Wir nutzen hier den aktuellen Arbeitsordner oder passen den Pfad an deine Umgebung an
BASE_DIR = Path(".")
DATA_DIR = BASE_DIR

ORIGINAL_RAW_PATH = DATA_DIR / "fraud_call_transcripts_curator.jsonl"
NOISY_RAW_PATH = DATA_DIR / "dialogues_transcripts_noisy_new.jsonl"

NOISY_RAW_PATH.parent.mkdir(parents=True, exist_ok=True)

# Wahrscheinlichkeiten für die Noise-Injektion (anpassbar)
PROB_DUPLICATE = 0.08
PROB_PII = 0.07         # Kumuliert bis 0.15
PROB_GARBAGE = 0.05     # Kumuliert bis 0.20
PROB_ENCODING = 0.03    # Kumuliert bis 0.23
# Rest (~77%) bleibt sauber

PII_SAMPLES = [
    "Meine Kreditkartennummer lautet 4532-8921-1029-4411 mit CVV 892.",
    "Sie erreichen mich unter max.mustermann@example.de oder Mobil: +49 171 1234567.",
    "Meine IBAN lautet DE89 3704 0044 0532 0130 00, Inhaber ist Thomas Müller.",
    "Ich wohne in der Hauptstraße 45, 10115 Berlin. Geburtsdatum ist der 14.05.1982.",
    "Meine Sozialversicherungsnummer is 12 140582 M 043."
]

GARBAGE_SNIPPETS = [
    "???",
    "N/A",
    "asdfghjkl;",
    "NULL",
    "[ERROR_VOICEMAIL_RECORDING_CORRUPTED]",
    "CLICK... BEEP... BEEP...",
    "a"
]

print("✅ Noise-Konfiguration geladen!")

In [ ]:
Zelle 6: Hilfsfunktionen für das Einlesen und Textformatieren
Diese Zelle enthält die Logik zum sicheren Einlesen der zuvor generierten JSONL-Datei.

In [ ]:
# ==============================================================================
# 2. HILFSFUNKTIONEN
# ==============================================================================
def load_jsonl(file_path: Path) -> List[Dict[str, Any]]:
    """Lädt eine JSONL-Datei zeilenweise und fängt Encoding- oder Parsing-Fehler ab."""
    records = []
    if not file_path.exists():
        return records
    
    with open(file_path, "r", encoding="utf-8") as f:
        for line_no, line in enumerate(f, 1):
            line_str = line.strip()
            if not line_str:
                continue
            try:
                records.append(json.loads(line_str))
            except json.JSONDecodeError as e:
                print(f"⚠️ Warnung: Ungültiges JSON in Zeile {line_no}: {e}")
    return records

def format_to_text(item: Dict[str, Any]) -> str:
    """Extrahiert den Fließtext oder baut das Transkript sicher zusammen."""
    if "text" in item and isinstance(item["text"], str):
        return item["text"]
    
    raw_response = item.get("response", "")
    if not raw_response:
        return ""
        
    try:
        parsed = json.loads(raw_response) if isinstance(raw_response, str) else raw_response
        if isinstance(parsed, dict) and "transcript" in parsed:
            return "\n".join([f"{t.get('speaker', 'Unbekannt')}: {t.get('text', '')}" for t in parsed["transcript"]])
    except (json.JSONDecodeError, TypeError):
        pass
        
    return ""

print("✅ Hilfsfunktionen definiert!")

In [ ]:
Zelle 7: Ausführung der Noise Injection
Diese Zelle führt die Verunreinigung der Daten durch und gibt die Statistik aus.

In [ ]:
# ==============================================================================
# 3. HAUPTLOGIK AUSFÜHREN
# ==============================================================================
def run_noise_injection():
    print(f"📖 Lade Original-Transkripte von: {ORIGINAL_RAW_PATH}")
    input_records = load_jsonl(ORIGINAL_RAW_PATH)

    if not input_records:
        print(f"❌ FEHLER: Keine Daten in '{ORIGINAL_RAW_PATH}' gefunden oder Datei existiert nicht! Bitte erst die Generierung ausführen.")
        return

    print(f"🧬 Injiziere Datenmüll, PII, Duplikate & Encoding-Fehler in {len(input_records)} Einträge...")
    noisy_dataset = []
    stats = {"duplicate": 0, "pii": 0, "garbage": 0, "encoding": 0, "clean": 0}

    for idx, item in enumerate(input_records):
        call_id = item.get("call_id", item.get("id", f"CALL_{idx:04d}"))
        flat_text = format_to_text(item)

        if not flat_text:
            continue

        dice = random.random()
        
        if dice < PROB_DUPLICATE:
            noisy_dataset.append({"call_id": call_id, "text": flat_text, "is_clean": True, "noise_type": "none"})
            noisy_dataset.append({"call_id": f"{call_id}_DUP", "text": flat_text, "is_clean": False, "noise_type": "exact_duplicate"})
            stats["duplicate"] += 1
            
        elif dice < (PROB_DUPLICATE + PROB_PII):
            pii_text = f"{flat_text}\nAnrufer: {random.choice(PII_SAMPLES)}"
            noisy_dataset.append({"call_id": call_id, "text": pii_text, "is_clean": False, "noise_type": "pii_injection"})
            stats["pii"] += 1
            
        elif dice < (PROB_DUPLICATE + PROB_PII + PROB_GARBAGE):
            noisy_dataset.append({"call_id": call_id, "text": random.choice(GARBAGE_SNIPPETS), "is_clean": False, "noise_type": "garbage_text"})
            stats["garbage"] += 1
            
        elif dice < (PROB_DUPLICATE + PROB_PII + PROB_GARBAGE + PROB_ENCODING):
            broken_text = flat_text.replace("ä", "Ã¤").replace("ö", "Ã¶").replace("ü", "Ã¼").replace("ß", "Ã\x9f")
            noisy_dataset.append({"call_id": call_id, "text": broken_text, "is_clean": False, "noise_type": "encoding_error"})
            stats["encoding"] += 1
            
        else:
            noisy_dataset.append({"call_id": call_id, "text": flat_text, "is_clean": True, "noise_type": "none"})
            stats["clean"] += 1

    print(f"💾 Schreibe verunreinigte Testdaten nach: {NOISY_RAW_PATH}")
    with open(NOISY_RAW_PATH, "w", encoding="utf-8") as f:
        for rec in noisy_dataset:
            f.write(json.dumps(rec, ensure_ascii=False) + "\n")

    print("\n📊 INJEKTIONS-STATISTIK:")
    print(f"  • Base Directory:                 {BASE_DIR.resolve()}")
    print(f"  • Original-Transkripte eingelesen: {ORIGINAL_RAW_PATH.name} ({len(input_records)} Zeilen)")
    print(f"  • Duplikate injiziert:            {stats['duplicate']}")
    print(f"  • PII-Einträge injiziert:         {stats['pii']}")
    print(f"  • Müll-Texte injiziert:           {stats['garbage']}")
    print(f"  • Encoding-Fehler erzeugt:        {stats['encoding']}")
    print(f"  • Saubere Texte belassen:         {stats['clean']}")
    print(f"\n✅ Erfolgreich ausgeführt! {len(noisy_dataset)} Einträge in '{NOISY_RAW_PATH.name}' erstellt.")

# Im Notebook direkt ausführen
run_noise_injection()

In [ ]:
Zelle 8: Imports & Umgebungsvariablen für Curation
Führe diese Zelle zuerst aus, um alle NeMo Curator Module, RAPIDS-Bibliotheken und die GDS/cuFile-Workarounds zu laden.

In [ ]:
import sys
import json
import re
import shutil
from pathlib import Path
from typing import Any
import os

# Native NeMo Curator & RAPIDS Imports
from nemo_curator import Modify, ScoreFilter, Sequential, AddId
from nemo_curator.datasets import DocumentDataset
from nemo_curator.filters import DocumentFilter
from nemo_curator.modules import ExactDuplicates
from nemo_curator.classifiers import DomainClassifier
from nemo_curator.utils.distributed_utils import get_client
from nemo_curator.modifiers import DocumentModifier, UnicodeReformatter

# Saubere Initialisierung, die den Segmentation Fault bei cuDF/Dask umgeht:
from dask_cuda import LocalCUDACluster
from distributed import Client

# ==============================================================================
# 1. PFAD- & ORDNER-KONFIGURATION IM NOTEBOOK
# ==============================================================================
BASE_DIR = Path(".")
DATA_DIR = BASE_DIR

INPUT_PATH = DATA_DIR / "dialogues_transcripts_noisy_new.jsonl"
CURATED_OUT_PATH = DATA_DIR / "dialogues_transcripts_curator_new.jsonl"
TEMP_EXPORT_DIR = DATA_DIR / "_temp_curator_export"
DEDUP_LOG_DIR = DATA_DIR / "dedup_logs"
DEDUP_CACHE_DIR = DATA_DIR / "dedup_cache"

CURATED_OUT_PATH.parent.mkdir(parents=True, exist_ok=True)
DEDUP_LOG_DIR.mkdir(parents=True, exist_ok=True)
DEDUP_CACHE_DIR.mkdir(parents=True, exist_ok=True)

# -- GDS / cuFile Workaround für Container-Umgebungen --
os.environ["KVIKIO_COMPAT_MODE"] = "ON"
os.environ["CUDF_CUFILE_ENABLED"] = "0"
os.environ["RAPIDS_NO_CUFILE"] = "1"

print("✅ Curation-Imports & Pfade erfolgreich initialisiert!")

In [ ]:
Zelle 9: Validierung, Custom Modifiers & Filters
Diese Zelle definiert die Schema-Prüfung, die PII-Maskierung (FraudPiiModifier) sowie den Längenfilter (MinLengthFilter).

In [ ]:
# ==============================================================================
# 2. STRIKTES ERROR HANDLING & VALIDIERUNG
# ==============================================================================
def validate_input_file(file_path: Path) -> int:
    print(f"🔍 Validiere Eingabedatei & Schema: {file_path.name}...")
    if not file_path.exists():
        print(f"❌ KRITISCHER FEHLER: Eingabedatei '{file_path}' existiert nicht! Bitte zuerst die Noise-Injection ausführen.")
        return 0

    total_lines = 0
    with open(file_path, "r", encoding="utf-8") as f:
        for line_no, line in enumerate(f, 1):
            line_str = line.strip()
            if not line_str:
                continue
            
            try:
                record = json.loads(line_str)
            except json.JSONDecodeError as e:
                print(f"❌ SCHEMAFEHLER [Zeile {line_no}]: Kein valides JSON. Fehler: {e}")
                return 0
            
            if "text" not in record:
                print(f"❌ SCHEMAFEHLER [Zeile {line_no}]: Pflichtfeld 'text' fehlt!")
                return 0
                
            total_lines += 1

    if total_lines == 0:
        print("❌ KRITISCHER FEHLER: Eingabedatei ist vollkommen leer!")
        return 0

    print(f"✅ Validation erfolgreich! {total_lines} Einträge für NeMo Curator bereit.\n")
    return total_lines

# ==============================================================================
# 3. CURATOR EXTENSIONS (PII-Maskierung & Filter)
# ==============================================================================
class FraudPiiModifier(DocumentModifier):
    """Custom Modifier zur Maskierung von PII (IBAN, Kreditkarten, E-Mails, Geburtsdaten, Handynummern, Adressen, Kundennummern)."""
    def modify_document(self, doc: str) -> str:
        if not isinstance(doc, str):
            return doc
        
        # 1. IBAN
        doc = re.sub(r'DE\d{2}\s?(\d{4}\s?){4}\d{2}', '[IBAN_MASKIERT]', doc)
        # 2. Kreditkarten
        doc = re.sub(r'\b(?:\d[ -]*?){13,16}\b', '[KREDITKARTE_MASKIERT]', doc)
        # 3. E-Mails
        doc = re.sub(r'[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\.[a-zA-Z]{2,}', '[EMAIL_MASKIERT]', doc)
        # 4. Geburtsdaten
        doc = re.sub(r'\b\d{1,2}\.\s+(?:Januar|Februar|März|April|Mai|Juni|Juli|August|September|Oktober|November|Dezember)\s+\d{4}\b', '[GEBURTSDATUM_MASKIERT]', doc, flags=re.IGNORECASE)
        # 5. Handynummern
        doc = re.sub(r'\b(?:\+49|0)\s*1[567]\d[\s\-]?\d{3,8}\b', '[HANDYNUMMER_MASKIERT]', doc)
        # 6. Adressen
        doc = re.sub(r'\b(?:[A-ZÄÖÜ][a-zäöüß]+\s+)?[A-ZÄÖÜ][a-zäöüß]+(?:straße|str\.|weg|allee|platz|ring)\s+\d+[a-zA-Z]?,?\s*\d{5}\s+[A-ZÄÖÜ][a-zäöüß]+\b', '[ADRESSE_MASKIERT]', doc, flags=re.IGNORECASE)
        # 7. Kundennummern
        doc = re.sub(r'\b\d{9}\b', '[KUNDENNUMMER_MASKIERT]', doc)
        
        return doc


class MinLengthFilter(DocumentFilter):
    def __init__(self, min_length: int = 50, text_field: str = "text"):
        super().__init__()
        self.min_length = min_length
        self.text_field = text_field

    def score_document(self, doc: Any) -> int:
        if isinstance(doc, dict):
            text = doc.get(self.text_field, "")
        elif hasattr(doc, self.text_field):
            text = getattr(doc, self.text_field, "")
        elif isinstance(doc, str):
            text = doc
        else:
            return 0
        
        return len(str(text).strip())

    def keep_document(self, score: int) -> bool:
        return score >= self.min_length

print("✅ Validierungs- und Erweiterungsklassen definiert!")

In [ ]:
Zelle 10: Ausführung der Advanced Curation Pipeline
Diese Zelle startet das GPU-gestützte Framework, filtert, maskiert PII, bereinigt Duplikate und generiert den finalen Statistikbericht.

In [ ]:
# ==============================================================================
# 4. HAUPT-PIPELINE IM NOTEBOOK AUSFÜHREN
# ==============================================================================
def run_curation_pipeline():
    initial_count = validate_input_file(INPUT_PATH)
    if initial_count == 0:
        return

    print("🚀 Initialisiere NeMo Curator Execution Client (GPU/CUDA Backend)...")
    client = get_client(cluster_type="gpu", set_torch_to_use_rmm=False)
    print("🔗 Dask-Client erfolgreich verbunden.")

    print("⚡ 1. Lade Dataset in NeMo Curator...")
    dataset = DocumentDataset.read_json(str(INPUT_PATH), add_filename=True, backend="pandas")

    print("🆔 2. Generiere eindeutige IDs für NeMo Curator...")
    add_id = AddId(id_field="id", id_prefix="FRAUD_data", start_index=0)
    dataset = add_id(dataset)

    print("🛡️ 3. Wende Cleaning-Sequenz an (Unicode & PII-Maskierung)...")
    cleaners = Sequential([
        Modify(UnicodeReformatter()),
        Modify(FraudPiiModifier())
    ])
    dataset = cleaners(dataset).persist()

    print("🧹 4. Wende NeMo Curator MinLengthFilter an (Min 50 Zeichen)...")
    length_filter = ScoreFilter(
        MinLengthFilter(min_length=50, text_field="text"),
        score_type=int
    )
    dataset = length_filter(dataset)

    print("✂️ 5. Führe Exakte Deduplizierung (ExactDuplicates) aus...")
    exact_dup = ExactDuplicates(
        logger=str(DEDUP_LOG_DIR),
        id_field="id",
        text_field="text",
        hash_method="md5",
        cache_dir=str(DEDUP_CACHE_DIR),
    )
    duplicates_dataset = exact_dup(dataset=dataset)
    
    # Identifiziere Duplikate, die entfernt werden sollen
    exact_docs_to_remove = duplicates_dataset.df.map_partitions(
        lambda x: x[x._hashes.duplicated(keep="first")]
    )

    # Herausfiltern der Duplikate
    id_field = "id"
    cleaned_df = dataset.df[
        ~dataset.df[id_field].isin(exact_docs_to_remove[id_field].compute())
    ]
    dataset = DocumentDataset(cleaned_df)

    # Aufräumen alter Temp-Ordner
    if TEMP_EXPORT_DIR.exists():
        shutil.rmtree(TEMP_EXPORT_DIR)
    if CURATED_OUT_PATH.exists():
        if CURATED_OUT_PATH.is_dir():
            shutil.rmtree(CURATED_OUT_PATH)
        else:
            CURATED_OUT_PATH.unlink()

    print(f"💾 Schreibe finalen, kurierten Datensatz nach: {CURATED_OUT_PATH}")
    dataset.to_json(str(TEMP_EXPORT_DIR), write_to_filename=False)

    # Zusammenführen der Partitions-Dateien
    exported_files = list(TEMP_EXPORT_DIR.glob("*.json*")) + list(TEMP_EXPORT_DIR.glob("*.part"))
    if exported_files:
        shutil.move(str(exported_files[0]), str(CURATED_OUT_PATH))
        shutil.rmtree(TEMP_EXPORT_DIR)
    else:
        with open(CURATED_OUT_PATH, "w", encoding="utf-8") as outfile:
            for part in sorted(TEMP_EXPORT_DIR.iterdir()):
                if part.is_file():
                    with open(part, "r", encoding="utf-8") as infile:
                        shutil.copyfileobj(infile, outfile)
        shutil.rmtree(TEMP_EXPORT_DIR)

    # Finale Auswertung
    final_docs = []
    with open(CURATED_OUT_PATH, "r", encoding="utf-8") as f:
        for line in f:
            if line.strip():
                final_docs.append(json.loads(line))
    
    final_count = len(final_docs)
    total_removed = initial_count - final_count

    print("\n" + "="*80)
    print("📊 NEMO CURATOR ADVANCED PIPELINE STATISTIK")
    print("="*80)
    print(f"  • Eingelesene Roh-Datensätze:      {initial_count}")
    print(f"  • Gefilterte / Duplizierte Einträge: {total_removed}")
    print("-" * 80)
    print(f"  • Verbliebene kurierte Datensätze: {final_count}")
    print(f"  • Verwerfungsquote Total:          {(total_removed / initial_count if initial_count > 0 else 0):.2%}")
    print("="*80)
    print(f"\n✅ Pipeline erfolgreich beendet! Saubere Datei: {CURATED_OUT_PATH.name}")

# Im Notebook direkt ausführen
run_curation_pipeline()

In [ ]:
Zelle 11: Konfiguration & Pfade für die SFT-Vorbereitung
Führe diese Zelle aus, um die Pfade für die kuratierten Daten, die Benchmark-Labels und den Ausgabeordner für das Supervised Fine-Tuning zu definieren.

Python

In [ ]:
import os
import sys
import json
import random
from pathlib import Path

# ==============================================================================
# 1. PFAD-KONFIGURATION IM NOTEBOOK
# ==============================================================================
BASE_DIR = Path(".")
DATA_DIR = BASE_DIR

CURATED_PATH = DATA_DIR / "dialogues_transcripts_curator_new.jsonl"
BENCHMARK_PATH = DATA_DIR / "fraud_call_benchmark_curator.jsonl"
SFT_DIR = DATA_DIR / "sft"

print("✅ SFT-Pfade erfolgreich konfiguriert!")

In [ ]:
Zelle 12: Hilfsfunktionen & Validierung
Diese Zelle enthält die Fail-Fast-Validierung sowie Funktionen zum Einlesen und Speichern von JSONL-Dateien.

In [ ]:
# ==============================================================================
# 2. STRIKTES ERROR HANDLING (FAIL-FAST) & HELPER
# ==============================================================================
def validate_required_files():
    print("🔍 Prüfe Eingabedateien...")
    
    if not CURATED_PATH.exists():
        print(f"❌ KRITISCHER FEHLER: Kuratierte Datei nicht gefunden: '{CURATED_PATH}'")
        return False
        
    if not BENCHMARK_PATH.exists():
        print(f"❌ KRITISCHER FEHLER: Benchmark-Datei nicht gefunden: '{BENCHMARK_PATH}'")
        return False

    print("✅ Alle benötigten Dateien wurden gefunden.")
    return True

def load_jsonl(path):
    records = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            if line.strip():
                records.append(json.loads(line))
    return records

def save_jsonl(records, path):
    with open(path, "w", encoding="utf-8") as f:
        for r in records:
            f.write(json.dumps(r, ensure_ascii=False) + "\n")

print("✅ SFT-Hilfsfunktionen definiert!")

In [ ]:
Zelle 13: Ausführung der SFT Data Preparation Pipeline
Diese Zelle verknüpft die kuratierten Transkripte mit den Ground-Truth-Labels, führt den stratifizierten 70/15/15-Split durch und exportiert die finalen train, validation und test JSONL-Dateien.

Python

In [ ]:
# ==============================================================================
# 3. MAIN PIPELINE FÜR SFT DATEN
# ==============================================================================
def run_sft_preparation():
    if not validate_required_files():
        return

    print("📊 Bereite SFT-Datensätze vor...")
    
    curated_data = load_jsonl(CURATED_PATH)
    benchmark_data = load_jsonl(BENCHMARK_PATH)

    # 1. Lookup-Map aus Benchmark aufbauen (Schlüssel: "id", z.B. "doc-00001")
    gt_map = {}
    for item in benchmark_data:
        cid = item.get("id")
        label = item.get("label") or item.get("fraud_type")
        if cid and label:
            gt_map[str(cid)] = label

    # 2. Matching über 'call_id' oder 'id' ausführen
    sft_samples = []
    matched_count = 0
    unmatched_count = 0

    for doc in curated_data:
        text = doc.get("text", "")
        cid = str(doc.get("call_id") or doc.get("id", ""))
        
        gt_label = gt_map.get(cid)
        
        if gt_label:
            matched_count += 1
            sft_samples.append({
                "input": text,
                "output": gt_label
            })
        else:
            unmatched_count += 1

    total_docs = len(curated_data)
    print(f"ℹ️ Total geladene kuratierte Dokumente: {total_docs}")
    print(f"    🔗 Erfolgreich mit Ground-Truth gematcht: {matched_count}")
    
    if unmatched_count > 0:
        print(f"    ⚠️ Ohne passendes Label übersprungen: {unmatched_count} Dokument(e)")

    if len(sft_samples) == 0:
        print(f"\n❌ KRITISCHER FEHLER: Es konnten keine einzigen SFT-Daten gematcht werden!")
        return

    # 3. Train / Val / Test Split (70% / 15% / 15%)
    random.seed(42)
    random.shuffle(sft_samples)

    total_samples = len(sft_samples)
    train_end = int(total_samples * 0.70)
    val_end = train_end + int(total_samples * 0.15)

    train_data = sft_samples[:train_end]
    val_data = sft_samples[train_end:val_end]
    test_data = sft_samples[val_end:]

    # 4. Speichern
    SFT_DIR.mkdir(parents=True, exist_ok=True)
    
    train_path = SFT_DIR / "train_new.jsonl"
    val_path = SFT_DIR / "validation_new.jsonl"
    test_path = SFT_DIR / "test_new.jsonl"

    save_jsonl(train_data, train_path)
    save_jsonl(val_data, val_path)
    save_jsonl(test_data, test_path)

    print(f"\n    -> Exportiert: train_new.jsonl ({len(train_data)} Einträge)")
    print(f"    -> Exportiert: validation_new.jsonl ({len(val_data)} Einträge)")
    print(f"    -> Exportiert: test_new.jsonl ({len(test_data)} Einträge)")
    print(f"\n✅ SFT-Datensätze erfolgreich in Ordner '{SFT_DIR}' gespeichert!")

# Im Notebook direkt ausführen
run_sft_preparation()